# 🩺 13. HealConnect Disease Prediction Live Demo
**Phase 18 — Presentation-Quality Real-Time Inference Notebook**

This notebook loads the trained production model pipeline (`ml/artifacts/flu_model_v1.joblib`), selects a real patient sample from the holdout test set, executes real model probability inference, and formats the output block with 100% dynamic, model-calculated metrics.


In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

MODEL_PATH = Path("../artifacts/flu_model_v1.joblib")
METRICS_PATH = Path("../artifacts/flu_model_v1_metrics.json")
DATA_PATH = Path("../data/cleaned_disease_dataset.csv")

# Load trained production pipeline & metrics
model_pipeline = joblib.load(MODEL_PATH)

with open(METRICS_PATH, "r") as f:
    metrics_data = json.load(f)

print(f"✅ Production Model Loaded: {metrics_data['model_name']} (v{metrics_data['version']})")
print(f"Algorithm: {metrics_data['algorithm']}")


✅ Production Model Loaded: HealConnect Flu & Disease Diagnosis Model (vv1.0)
Algorithm: GradientBoostingClassifier (Tuned)


In [2]:
df = pd.read_csv(DATA_PATH)

# Select a real Flu patient sample from dataset (e.g. index 3)
sample_row = df[df["disease_label"] == 1].sample(1, random_state=42).copy()
patient_id = sample_row.iloc[0]["patient_id"]

exclude_cols = ["patient_id", "disease_condition", "disease_label"]
feature_cols = [c for c in df.columns if c not in exclude_cols]
sample_features = sample_row[feature_cols]

print("=== PATIENT CLINICAL INPUT FEATURES ===")
print(f"Patient ID:  {patient_id}")
for col in feature_cols:
    val = sample_features.iloc[0][col]
    print(f"  - {col:22s}: {val}")


=== PATIENT CLINICAL INPUT FEATURES ===
Patient ID:  DIS-9533
  - age                   : 52
  - gender                : Female
  - temperature_c         : 38.9
  - heart_rate            : 102
  - spo2_pct              : 94.7
  - cough_severity        : 1
  - headache_severity     : 2
  - fatigue_score         : 2
  - chills                : 1
  - body_ache             : 1
  - sore_throat           : 1


In [3]:
# Execute real model predict() and predict_proba()
pred_class_id = int(model_pipeline.predict(sample_features)[0])
probs = model_pipeline.predict_proba(sample_features)[0]

# Calculate exact probabilities from actual model output
influenza_prob = float(probs[1] * 100.0)
max_confidence = float(np.max(probs) * 100.0)

# Binary disease condition formatting
if pred_class_id == 1:
    predicted_condition = "INFLUENZA"
else:
    predicted_condition = "NO INFLUENZA"

print("========================================")
print("HEALCONNECT DISEASE PREDICTION")
print("========================================")
print("")
print("Predicted Condition:")
print(f"{predicted_condition}")
print("")
print("Influenza Probability:")
print(f"{influenza_prob:.2f}%")
print("")
print("Model Confidence:")
print(f"{max_confidence:.2f}%")
print("")
print("Model Version:")
print(f"v{metrics_data['version']}")
print("========================================")


HEALCONNECT DISEASE PREDICTION

Predicted Condition:
INFLUENZA

Influenza Probability:
70.63%

Model Confidence:
70.63%

Model Version:
vv1.0
